In [ ]:
import scanpy as sc
import rpy2.robjects as ro
from scipy.stats import median_abs_deviation as mad
from pathlib import Path
from src import *

%load_ext autoreload
%autoreload 2
%load_ext rpy2.ipython

sc.set_figure_params(dpi=150, facecolor='white')
sc.settings.verbosity = 3

project_root = Path.cwd().parent
data_dir = project_root / "data"
results_dir = project_root / "results"
ro.r(f'renv::load("{project_root}")')

In [ ]:
adata = sc.read_h5ad(data_dir / "processed" / "00_combined_processed.h5ad")
mesenchymal = adata[adata.obs["lineage"] == "mesenchymal"].copy()

In [ ]:
mesenchymal = deviant_hvg(mesenchymal)

In [ ]:
plot_deviance(mesenchymal)

In [ ]:
harmony_embed(mesenchymal)
sc.pl.pca_overview(mesenchymal, color="sample_id")

In [ ]:
cluster_subset(mesenchymal, n_pcs=20, resolution=0.5)

In [ ]:
sc.pl.umap(mesenchymal, color=["scDblFinder_score", "sample_id", "pct_counts_mt", "leiden"], ncols=2)

In [ ]:
mesenchymal_markers = {
    "fibroblast": ["Col1a1", "Col1a2", "Col3a1", "Pdgfra", "Dcn", "Npnt", "Serpinf1", "Mfap5", "Ogn"],
    "smooth muscle": ["Acta2", "Tagln"],
    "pericyte": ["Pdgfrb", "Cspg4", "Rgs5"],
    "mesothelium": ["Msln", "Gpm6a"],
    "proliferating cells": ["Mki67", "Top2a"],
    "neural": ["Sox10", "Mpz"],
    "pvcms": ["Tnnt2"]
}

sc.tl.dendrogram(mesenchymal, groupby="leiden", use_rep="X_pca_harmony")
sc.pl.matrixplot(
    mesenchymal,
    var_names=mesenchymal_markers,
    groupby="leiden",
    use_raw=False,
    dendrogram=True,
    cmap="Reds",
    standard_scale="var",
)

In [ ]:
mesenchymal_celltypes = {
    "fibroblast": ["5", "1", "13", "3", "4", "7"],
    "fibroblast-like": ["8"],
    "smooth muscle": ["10"],
    "proliferating smooth muscle": ["11"],
    "pericyte": ["0"],
    "proliferating pericyte": ["12"],
    "mesothelium": ["6"],
    "proliferating fibroblasts": ["2"],
    "neural": ["14"],
    "pvcms": ["9"]
}

apply_celltypes(mesenchymal, mesenchymal_celltypes)

In [ ]:
sc.pl.umap(mesenchymal, color="celltype", legend_loc="on data", legend_fontsize=6)

In [ ]:
mesenchymal.write_h5ad(data_dir / "processed" / "04_mesenchymal.h5ad")